# Which images does Ramen confuse, and with what? Per-image records on the full DomainNet

**Question.** When the zero-shot model or Ramen gets an image wrong, which class does it pick instead? What do
those images look like? And how does one adaptation step change the image's embedding and prediction?

**What this notebook is.** This is a tool for looking at errors, not a test of a new method. Nothing here
changes Ramen. Every image is run through `RamenRecorder` (`src/methods/RamenRecorder.py`). This is Ramen with
one addition: it records four things for each image:

| record | what it is |
|---|---|
| `zs_probs` | zero-shot softmax over the 345 classes, from the pass before adaptation (this pass also decides which memory queue the image is filed in) |
| `probs` | Ramen's softmax, after its one signSGD step on the norm parameters |
| `emb_before` | L2-normalised CLIP image embedding with the pretrained norm parameters (= the zero-shot embedding) |
| `emb_after` | the same embedding with the adapted norm parameters (the one Ramen classifies) |

The `zero_shot → ramen` **outcome** of each image is one of: right → right, *fixed*, *broken*,
wrong → same wrong, wrong → other wrong. The **embedding shift** is 1 − cos(before, after).

The class confusion of each source is a 345 × 345 count matrix: rows are the true class, columns the predicted
class (`confusion.py`). For one class *c* over the N images:
- TP: true *c*, predicted *c*
- FN: true *c*, predicted something else
- FP: true something else, predicted *c*
- TN: everything else

**Data and model.** DomainNet is a **full run** on all **596,006 images** in 6 domains (clipart, infograph,
painting, quickdraw, real, sketch) and **345 classes**. Model: CLIP ViT-B/32. Ramen config:
`cfg/DomainNet/Ramen.yaml` (capacity 300 per class, top-k 10, β 5, lr 0.01, signSGD, fp16 priorities as in Ramen).
One mixed, shuffled stream, batch size 100. **Seed 0** sets the stream order. Known baseline, seed 0: NoAdapt
55.37 → Ramen 56.80 (domain average). Seeds 0–4 give a noise floor of 0.11. This is the same stream as the cached
Ramen run, so these are the full-run numbers; notebook 01 is the 12,000-image subset.

**How the results were produced.**
```
cd new_experiments/2026-10-28_confusion_records
CUDA_VISIBLE_DEVICES=0 nice -n 10 python run_records.py DomainNet 0     # ~40 min, ~2 GB of vectors
```
The run is cached under `new_experiments/results/DomainNet_clip_vitbase32_mixed_nall_bs100_s0_rec_Ramen*`.
Code: `experiment.py` (RecordExperiment), `records.py` (ImageRecords), `confusion.py` (ConfusionMatrix,
ConfusionShift), `gallery.py` (ConfusionGallery), `plots.py`.

**Sanity check (what counts as valid).** The recorder must not change Ramen. Its predictions must equal those of
plain Ramen on the same stream for 100.00% of images (cell 2). The recorded zero-shot prediction is compared with
the separate NoAdapt run for information only. The two use different model wrappers, so fp16 near-ties can flip a
handful of images.

In [ ]:
import sys
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from experiment import RecordExperiment, count_pct
import plots

pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

# ---- settings: change these to look at something else -------------------------------------------------
DATASET, SEED = 'DomainNet', 0
N_PER_DOMAIN = None                      # None = the whole dataset; an int = that many images per domain
FOCUS_CLASS = 'dog'                        # one class to inspect in detail
FOCUS_PAIR = ('snake', 'squiggle')                          # one (true, predicted) confusion to inspect
OLD_SINK = 'squiggle'                              # a class zero-shot over-predicts
NEW_SINK = 'pillow'                              # a class Ramen over-predicts more than zero-shot does

In [ ]:
exp = RecordExperiment(DATASET, seed=SEED, n_per_domain=N_PER_DOMAIN).run()   # cached -> no GPU needed
assert exp.sanity_check(), 'the recorder changed Ramen'
records = exp.records
zs, ra = exp.confusion('zero_shot'), exp.confusion('ramen')
shift = exp.shift()
print(f'{len(records):,} images recorded; vectors: ' +
      ', '.join(f'{k} {tuple(v.shape)} {v.dtype}' for k, v in records.arrays.items()))

## The data

Images and classes present per domain. These are the denominators for everything below. The vectors are memory-mapped from the HDD and read only when needed.

In [ ]:
display(exp.data_table())
support = records.table['label'].value_counts()
print(f'images per class: median {support.median():.0f}, min {support.min()}, max {support.max()}')

## Headline: zero-shot vs Ramen

Accuracy per domain, with counts. Below that, each image's outcome from zero-shot to Ramen. The net gain is
*fixed − broken*.

In [ ]:
display(exp.accuracy_table())
ot = exp.outcome_table(by='domain')
display(ot)
n = len(records)
for o in ['fixed', 'broken', 'wrong -> same wrong', 'wrong -> other wrong']:
    print(f'{o:22s} {count_pct(int(ot.loc["all", o]), n)}')
plots.plot_outcomes(ot.drop('all'), title='Quickdraw: 75,958 images switch to another wrong class, 10,521 are fixed')
plt.show()

## 1. Per-class confusion: TP, FP, FN, TN

`ConfusionMatrix.per_class()` lists every class. The interesting classes are **sinks**: classes predicted far more
often than they occur, which shows up as a large FP. The table below compares zero-shot and Ramen on the classes
whose FP changes most.

In [ ]:
pc = shift.per_class()
pc['predicted zero_shot'] = zs.counts.sum(0); pc['predicted ramen'] = ra.counts.sum(0)
cols = ['support', 'predicted zero_shot', 'predicted ramen', 'TP zero_shot', 'TP ramen', 'FP zero_shot', 'FP ramen', 'ΔFP',
        'FN zero_shot', 'FN ramen', 'ΔFN']
print('Classes whose false positives Ramen REMOVES most (zero-shot sinks):')
display(pc.sort_values('ΔFP')[cols].head(8))
print('Classes whose false positives Ramen ADDS most (new sinks):')
display(pc.sort_values('ΔFP', ascending=False)[cols].head(8))

In [ ]:
print('Ramen, per class, sorted by FP (TN is large for every class: N minus the rest):')
display(ra.per_class().sort_values('FP', ascending=False).head(10))

### Sink fate and sink origin

**Sink fate.** Take the images zero-shot wrongly sends to `OLD_SINK`. Does Ramen fix them, leave them there, or
move them to another wrong class?

**Sink origin.** Take Ramen's false positives of `NEW_SINK`. What did zero-shot predict for them?

In [ ]:
fate, moved = exp.sink_fate(OLD_SINK)
display(fate.to_frame(f'{OLD_SINK}: images'))
display(moved.to_frame('images'))
n_fp, origin, domains = exp.sink_origin(NEW_SINK)
print(f'Ramen predicts {NEW_SINK} wrongly for {n_fp:,} images (zero-shot: {int(zs.fp[zs._idx(NEW_SINK)]):,})')
display(origin.to_frame('images')); display(domains.head(6).to_frame('images'))

## 2. One class in detail

Set `FOCUS_CLASS` in the settings cell. Three views:
1. its TP / FN / FP / TN under both sources;
2. where its images go (`predicted_as`, the row of the matrix);
3. which classes get called by its name (`mistaken_for`, the column).

Then the images themselves. Each caption shows the true class, then the zero-shot prediction with its probability
and the probability of the true class, then the same for Ramen, then the domain and the embedding shift.

In [ ]:
display(pd.concat({'zero-shot': zs.report(FOCUS_CLASS)[['count', 'of', '%']],
                   'Ramen': ra.report(FOCUS_CLASS)[['count', 'of', '%']]}, axis=1))
plots.plot_class_report(shift, FOCUS_CLASS); plt.show()

In [ ]:
display(pd.concat({'zero-shot': zs.predicted_as(FOCUS_CLASS, 8).reset_index(),
                   'Ramen': ra.predicted_as(FOCUS_CLASS, 8).reset_index()}, axis=1))
display(pd.concat({'zero-shot': zs.mistaken_for(FOCUS_CLASS, 8).reset_index(),
                   'Ramen': ra.mistaken_for(FOCUS_CLASS, 8).reset_index()}, axis=1))

In [ ]:
exp.gallery.show(true=FOCUS_CLASS, correct=False, n=16, title=f'Images of {FOCUS_CLASS} that Ramen gets wrong '
                 f'(most confident first)'); plt.show()
exp.gallery.show(pred=FOCUS_CLASS, correct=False, n=8, title=f'Images Ramen wrongly calls {FOCUS_CLASS}'); plt.show()

## 3. Confused pairs

These are the largest off-diagonal cells under each source, then the pairs that change most from zero-shot to Ramen.

In [ ]:
display(pd.concat({'zero-shot': zs.top_pairs(12), 'Ramen': ra.top_pairs(12)}, axis=1))

In [ ]:
changes = pd.concat([shift.pair_changes(10, 'shrank'), shift.pair_changes(10, 'grew')]).drop_duplicates()
plots.plot_pair_changes(changes.iloc[::-1].reset_index(drop=True), title='Ramen removes squiggle / envelope confusions and sends rectangles to pillow'); plt.show()

In [ ]:
true, pred = FOCUS_PAIR
n_zs, n_ra = zs.pair(true, pred), ra.pair(true, pred)
print(f'{true} -> {pred}: zero-shot {n_zs:,}, Ramen {n_ra:,} (of {int(zs.counts[zs._idx(true)].sum()):,} images of {true})')
exp.gallery.show(true=true, pred=pred, source='zero_shot', n=16, order='confident',
                 title=f'Zero-shot calls these {true} images {pred}; the caption shows what Ramen does with them'); plt.show()

In [ ]:
exp.gallery.show(pred=NEW_SINK, correct=False, outcome=['broken', 'wrong -> other wrong'], n=16,
                 title=f'Images Ramen newly sends to {NEW_SINK} (zero-shot predicted something else)'); plt.show()

## 4. Embeddings before and after adaptation

One step on the norm parameters moves each image's embedding a little. The boxes show how far images move, grouped
by outcome. The PCA plot draws an arrow from each image's embedding before adaptation to its embedding after, for
the images of the focus pair's two classes.

In [ ]:
display(exp.embedding_shift_table())
if records.table['domain'].nunique() > 1:
    print('Median shift per domain and outcome (domains move by different amounts, so compare within a row):')
    display(exp.embedding_shift_by_domain())
plots.plot_embedding_shift(records); plt.show()
plots.plot_embedding_moves(records, list(dict.fromkeys(FOCUS_PAIR))); plt.show()

## 5. One image

`gallery.show_image(i)` shows the image with its top classes under zero-shot and under Ramen. The ★ marks the true
class. `records[i]` returns the raw `ImageRecord`, which holds both probability vectors and both embeddings.

In [ ]:
i = int(records.where(outcome='fixed', domain='quickdraw').index[0])
exp.gallery.show_image(i); plt.show()
j = int(records.where(outcome='broken').sort_values('zs_conf', ascending=False).index[0])
exp.gallery.show_image(j); plt.show()
r = records[i]
print(r.true_class, r.zs_class, '->', r.pred_class, '| emb_before', r.emb_before.shape, '| top-3 Ramen', r.top('ramen', 3, records.classes))

## Interpretation (seed 0, full DomainNet, 596,006 images)

**The subset's picture holds at full scale: Ramen mostly relocates errors.** Zero-shot gets 314,840 of 596,006
right and Ramen gets 324,966: 55.37 → 56.80 domain-averaged, +1.43, far above the 0.11 noise floor. Ramen fixes
20,428 images (3.4%) and breaks 10,302 (1.7%). Another 99,028 (16.6%) switch from one wrong class to another, and
75,958 of those are quickdraw. In quickdraw, 7 images switch wrong class for every 1 that gets fixed (75,958 vs
10,521).

**Zero-shot's squiggle sink is relocated, not repaired.** Zero-shot wrongly calls 44,623 images *squiggle*. Ramen
fixes 3,613 of them (8.1%), leaves 11,331 at squiggle, and moves 29,679 (66.5%) to other wrong classes: boomerang
1,848, pillow 1,248, teapot 1,005, rake 990, pear 934, hockey_stick 923. *envelope* is worse: of its 14,586 false
positives, Ramen fixes 800 (5.5%) and moves 9,634, with 3,541 going to *pillow*. Squiggle also loses 254 of its own
939 correct images.

**Pillow is the main new sink, and it is fed by envelope.** Ramen wrongly predicts pillow 6,484 times, against
1,147 for zero-shot. Of those 6,484, 3,570 were zero-shot *envelope* predictions and 1,248 *squiggle*; 6,129 are
quickdraw. The pairs that grow most all end in pillow: square, door, map, laptop, keyboard and picture_frame
(+151 to +243 images each). These are rectangle drawings. Teapot (+3,302 false positives), helmet (+1,972) and
toaster (+1,942) grow the same way, mainly from squiggle and pear. Ramen's median confidence on its new false
positives is 0.30–0.49, against 0.97 for its correct predictions.

**dog, at full scale.** 2,609 images. Zero-shot gets 1,836 right and Ramen 1,776 (−60). Ramen also cuts dog's
false positives from 742 to 511. Squiggle and envelope dominate the error picture, so a single animal class
changes little.

**Embedding shift: the pooled difference is mostly domain mix.** Pooled over all images, the median shift is 0.050
for fixed images and 0.030 for images that stay right. But quickdraw moves the most (median 0.056 even for
right → right) and supplies most of the fixed and switched images. Within each domain, fixed images move only
0.001–0.008 more than images that stay right (table above). On the 12,000-image subset, images that stay right
move *most* in every domain. That disagreement is not explained. One guess, untested: a short stream fills the
memory differently, so the retrieved gradients differ. Either way, the size of the shift does not separate helpful
steps from harmful ones on DomainNet.

**Caveats.** Seed 0 only. These are counts from one stream order. The headline gain is the cached Ramen run's.

## Glossary

- **Zero-shot (NoAdapt)**: CLIP classifies by cosine similarity between the image embedding and the text embedding
  of each class name. No adaptation.
- **Ramen**: test-time adaptation. Each image is written into the memory queue of its zero-shot class, together
  with its own entropy gradient. Gradients of the nearest stored images are retrieved, one signSGD step is taken
  on the LayerNorm parameters, the image is predicted, and the parameters are reset.
- **Embedding before / after**: the L2-normalised CLIP image feature with the pretrained / adapted norm parameters.
- **Embedding shift**: 1 − cosine(before, after). 0 means the step did not move the image.
- **TP / FP / FN / TN** (for class *c*): true and predicted *c* / predicted *c* but true other / true *c* but
  predicted other / neither.
- **Precision / recall / F1** (for class *c*): TP / predicted *c*; TP / true *c*; 2TP / (2TP + FP + FN), which is 0
  when TP is 0.
- **Sink**: a class predicted far more often than it occurs (large FP).
- **Sink fate**: what Ramen does with the images zero-shot wrongly sends to a sink. They can be fixed, stay at the
  sink, or move to another wrong class.
- **Sink origin**: for Ramen's false positives of a class, what zero-shot had predicted for them.
- **Outcome**: zero-shot → Ramen correctness of one image. *fixed* = wrong → right; *broken* = right → wrong.
- **Subset (`n_per_domain`)**: a random sample of images per domain, which makes a shorter stream. Ramen's memory
  differs from the full run, so its accuracy is not comparable to the full run.
- **Domain average**: the mean of the per-domain accuracies (the headline convention in this project).